# Session 04 · Part 1 — Prompt and Skill with Deep Agents

You build a **Research Brief Agent** with [`create_deep_agent`](https://docs.langchain.com/oss/python/deepagents/overview):
a structured system prompt, one web-search tool and one **Skill** that the agent loads only when a request needs it.

1. Write the system prompt in named sections.
2. Write the Skill `research-brief` (`SKILL.md` + a template in `references/`).
3. Look at what the model really receives on each call.
4. Ask a quick question (the Skill should stay unread), then ask for a brief (the Skill should be read).

Docs: [Context engineering in Deep Agents](https://docs.langchain.com/oss/python/deepagents/context-engineering) ·
[Skills](https://docs.langchain.com/oss/python/deepagents/skills)

## 0. Setup

In [1]:
import logging
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
for noisy in ("httpx", "httpx2"):  # HTTP clients log every request at INFO
    logging.getLogger(noisy).setLevel(logging.WARNING)
logger = logging.getLogger(__name__)

# ── Keys and model (.env of this session folder) ─────────────────────────────
load_dotenv()
for name in ("OPENAI_API_KEY", "TAVILY_API_KEY"):
    if not os.getenv(name, "").strip():
        raise RuntimeError(f"Missing {name}: copy .env.example to .env in the session folder and fill it in")
model = init_chat_model(os.getenv("MODEL", "openai:gpt-6-luna"))
logger.info("Model: %s", model.model_name)

OUT = Path("output")  # everything the agent writes goes here (git-ignored)
OUT.mkdir(exist_ok=True)

INFO | Model: gpt-6-luna


## 1. A tool and a way to watch the agent
`notebooks/utils.py` holds the helpers every notebook of this session uses:
- `internet_search` calls Tavily. The docstring is part of the prompt: name, description and argument descriptions are
  sent to the model with every request.
- `internet_search` logs one line per call with the normal `logger`: `internet_search('…') -> ok 1840 ms`.
- `stream()` shows the agent's output while it is produced: `agent.astream(..., stream_mode=["updates", "messages"])`
  prints one line per tool call (`→`) and tool result (`←`), and the answer token by token. Logging is diagnostics for
  the developer; streaming is the agent's output to the user.
- `ContextLog` (callback) records the **final** request of each model call and shows it like `/context` in Claude Code:
  the system prompt split into its parts, every tool schema, messages by kind, and the free space of the context window.
  The total is the input token count **the provider reports** for that call. System prompt and messages are counted with
  the tokenizer (`o200k_base`); the provider sends tool schemas in its own compact format, so they get the rest of the total.

In [2]:
from utils import SKILLS_DIR, ask, context_window, internet_search, routes, system_prompt, tool_calls  # notebooks/utils.py

WINDOW = context_window(model)  # max input tokens, from the model's LangChain profile
logger.info("Context window: %s tokens", f"{WINDOW:,}")

INFO | Context window: 1,050,000 tokens


## 2. The system prompt, in named sections
The structure follows Anthropic ([Effective context engineering](https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents)):
a one-line role, then `<background_information>`, `<instructions>` and `<output_description>`, separated by XML tags.
Plain wording (no CRITICAL / MUST), no "think step by step" (the model reasons on its own), and no examples
(add a few varied ones only if a failure needs them). It stays short: the procedure for writing a brief goes into
the Skill, which is loaded only when needed.

The prompt lives in one file, `prompts/system_prompt.md`, used by every part. It names no tool and no path: tool
descriptions say when to use each tool, and Deep Agents appends the skills index and the memory guidelines itself.
The only paths it needs come from the backend layout below: `system_prompt(LAYOUT)` adds a `<files>` section built
from the same table.

In [3]:
print(system_prompt())  # prompts/system_prompt.md, shared by the four parts

You are a research assistant for a city energy team.

<background_information>
The team reuses your answers in public reports, so "no reliable source found" is better than a guess.
</background_information>

<instructions>
- A task is done when every claim in the answer has a source.
- Prefer official agencies and research institutes. Cite only urls that a tool returned.
- Search and read freely. Write files only in the folders listed under <files>.
- If the request is unclear, ask one short question.
</instructions>

<output_description>
Answer quick questions in 1 to 3 sentences. Put each source url in brackets after the sentence it supports.
</output_description>


## 3. The Skill `research-brief`
A Skill is a folder with a `SKILL.md` (YAML frontmatter `name` + `description`, then instructions) and optional files.
The agent sees only `name` and `description` at startup (level 1), reads `SKILL.md` when a request matches (level 2) and
reads `references/brief_template.md` when the instructions say so (level 3).

In [4]:
for path in sorted(SKILLS_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(SKILLS_DIR))
print()
print((SKILLS_DIR / "research-brief" / "SKILL.md").read_text())

research-brief/SKILL.md
research-brief/references/brief_template.md

---
name: research-brief
description: Write a one-page research brief with cited web sources on a technical or market topic. Use when the user asks for a brief, a summary report or a comparison document; not for quick one-line factual questions.
---
# research-brief

## Steps
1. Run `internet_search` at most 3 times with different, focused queries; keep at most 4 sources.
2. Read `references/brief_template.md` (next to this file) and follow its sections exactly.
3. End every bullet in "Key findings" with the url of its source in brackets.
4. Save the brief with `write_file` as `brief.md` in the deliverables folder listed under <files> in your instructions, then reply with a 3-line summary and the file path.

## Rules
- Cite only urls that appeared in a search result.
- If sources disagree, say so under "Open questions".



## 4. The agent
`create_deep_agent` adds the file tools (`ls`, `read_file`, `write_file`, `edit_file`, `delete`, `glob`, `grep`) and the
`task` tool on top of ours. `skills=["/skills/"]` makes `SkillsMiddleware` list every skill of that folder in the system prompt.

Where the files live is decided by a **backend**; it is explained in Part 3. For now: `/skills/` is read from the
`skills/` folder on disk and `/workspace/` is written to `output/workspace-part1/`.

In [5]:
from deepagents import create_deep_agent
from deepagents.backends import CompositeBackend, FilesystemBackend, StateBackend
from langgraph.checkpoint.memory import InMemorySaver

WORKSPACE = OUT / "workspace-part1"
WORKSPACE.mkdir(exist_ok=True)

# path the agent sees -> (where the files are, what the agent keeps there); explained in Part 3
LAYOUT = {
    "/skills/": (FilesystemBackend(root_dir=SKILLS_DIR.resolve(), virtual_mode=True), None),  # listed by SkillsMiddleware
    "/workspace/": (FilesystemBackend(root_dir=WORKSPACE.resolve(), virtual_mode=True), "deliverables the user will open"),
}

agent = create_deep_agent(
    model=model,
    tools=[internet_search],
    system_prompt=system_prompt(LAYOUT),
    skills=["/skills/"],
    backend=CompositeBackend(default=StateBackend(), routes=routes(LAYOUT)),  # other paths: scratch files of one thread
    checkpointer=InMemorySaver(),
)

def chat(question: str, thread_id: str):
    """Stream one turn, then print one context row per model call."""
    return ask(agent, question, thread_id, window=WINDOW)

## 5. A quick question: what does the model receive?
<img src="images/b4_model_request.png" alt="what the model receives" width="640">

The system prompt the model receives is **our prompt + the skills index** added by `SkillsMiddleware`
(only `name` and `description`). Deep Agents 0.7 adds no other base prompt; the usage of each tool is in its description.

In [6]:
result, log = chat("What refrigerant do most new heat pumps sold in Europe use? One sentence.", "quick-1")
print("tools called:", tool_calls(result), "\n")
log.show()  # the last call, part by part

→ internet_search({"query": "European heat pump sales refrigerant most common R32 new heat pumps Europe official report", "max_r…)


INFO | internet_search('European heat pump sales refrigerant most common R32 new heat pumps E…) -> ok 3574 ms


← internet_search success: [{"title": "Heat Pumps in the European Union - JRC Publications", "url": "https://publications.jrc.ec.europa.e…
→ internet_search({"query": "2024 Europe heat pump sales refrigerant share R32 propane new units most common refrigerant Europe …)


INFO | internet_search('2024 Europe heat pump sales refrigerant share R32 propane new units m…) -> ok 2520 ms


← internet_search success: [{"title": "Heat Pumps in the European Union - JRC Publications", "url": "https://publications.jrc.ec.europa.e…
There isn’t one refrigerant across all types: most air-to-air heat pumps use R32, while air-to-water monoblocs are rapidly switching to propane (R290). [https://publications.jrc.ec.europa.eu/repository/bitstream/JRC144191/JRC144191_01.pdf]
call messages   tokens  in this call
   1        1    2,681  skills index
   2        3    3,432  skills index
   3        5    4,266  skills index
tools called: ['internet_search', 'internet_search'] 

Call 3/3: 4,266 of 1,050,000 tokens (0.41%, provider count)
  [██████▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒]  █ system prompt  ▓ tool schemas  ▒ messages
  System prompt                         623
    your system_prompt                  156
    skills index                        467
  Tool schemas (9)                    2,042
    ls 82 · read_file 329 · write_file 134 · edit_file 196 · delete 112 · glob 335 · grep 43

In [7]:
print(log.system)  # the full text of the system prompt above

You are a research assistant for a city energy team.

<background_information>
The team reuses your answers in public reports, so "no reliable source found" is better than a guess.
</background_information>

<instructions>
- A task is done when every claim in the answer has a source.
- Prefer official agencies and research institutes. Cite only urls that a tool returned.
- Search and read freely. Write files only in the folders listed under <files>.
- If the request is unclear, ask one short question.
</instructions>

<output_description>
Answer quick questions in 1 to 3 sentences. Put each source url in brackets after the sentence it supports.
</output_description>

<files>
- /workspace/ deliverables the user will open
</files>

## Skills System

You have access to a skills library that provides specialized capabilities and domain knowledge.

**Skills Skills**: `/skills/` (higher priority)

Sources labeled "Deepagents" are specific to this agent tool; sources labeled "Agents" are shar

## 6. A brief: the Skill is read, then its template
The request matches the skill's description, so the agent reads `SKILL.md` (level 2), then the template (level 3),
searches, and saves the artifact to `/workspace/brief.md` (on disk: `output/workspace-part1/brief.md`).
Watch `tokens` grow on every call: each tool result stays in the thread.

In [8]:
result, log = chat("Write a brief on heat pumps in cold climates for our city energy team.", "brief-1")
calls = tool_calls(result)
print("\ntools called:", calls)
read = [tc["args"].get("file_path") for m in result["messages"] if m.type == "ai" for tc in m.tool_calls if tc["name"] == "read_file"]
print("files read:", read, "\n")
log.show()  # the last call: tool calls and results now dominate the messages

→ read_file({"file_path": "/skills/research-brief/SKILL.md", "offset": 0, "limit": 1000})
← read_file success: @@ lines 1-15 of 15 @@
---
name: research-brief
description: Write a one-page research brief with cited web so…
→ read_file({"file_path": "/skills/research-brief/references/brief_template.md", "offset": 0, "limit": 1000})
→ internet_search({"query": "DOE cold climate heat pumps performance field study cold temperatures official", "max_results": 4})
→ internet_search({"query": "NREL cold climate air source heat pumps cold climate field performance research", "max_results": 4})
→ internet_search({"query": "IEA heat pumps cold climates policy electricity grid buildings cold weather", "max_results": 4})
← read_file success: @@ lines 1-15 of 15 @@
# <Topic>: research brief

**Audience:** <who will read it>

## Question
One sentence: …


INFO | internet_search('DOE cold climate heat pumps performance field study cold temperatures…) -> ok 1321 ms


← internet_search success: [{"title": "NREL Field Validation of Air-Source Heat Pumps for Cold Climates 2021-2023 | Heat Pump Database", …


INFO | internet_search('NREL cold climate air source heat pumps cold climate field performanc…) -> ok 1876 ms


← internet_search success: [{"title": "NREL Field Validation of Air-Source Heat Pumps for Cold Climates 2021-2023 | Heat Pump Database", …


INFO | internet_search('IEA heat pumps cold climates policy electricity grid buildings cold w…) -> ok 3336 ms


← internet_search success: [{"title": "Weather in cold climate regions", "url": "https://www.weatherapi.com/", "content": "{'location': {…
→ write_file({"file_path": "/workspace/brief.md", "content": "# Heat pumps in cold climates: research brief\n\n**Audience:*…)
← write_file success: Updated file /workspace/brief.md
Cold-climate heat pumps can save energy in many homes, though results vary by building and conditions. [https://research-hub.nlr.gov/en/publications/field-validation-of-air-source-heat-pumps-for-cold-climates]  
Field studies show performance changes with outdoor temperature and support piloting and tracking local outcomes before scaling. [https://www.pnnl.gov/main/publications/external/technical_reports/PNNL-37127.pdf]  
Brief saved to `/workspace/brief.md`.
call messages   tokens  in this call
   1        1    2,680  skills index
   2        3    2,947  skills index
   3        8    4,917  skills index
   4       10    6,071  skills index

tools called: ['read_file', 'r

In [9]:
brief = (WORKSPACE / "brief.md").read_text()
print(brief)
for section in ("## Question", "## Key findings", "## Open questions", "## Sources"):
    print(f"{section:20} {'ok' if section in brief else 'MISSING'}")

# Heat pumps in cold climates: research brief

**Audience:** City energy team

## Question
What does current field evidence suggest about heat-pump performance in cold climates, and what should the city consider when planning programs?

## Key findings
- Recent variable-capacity air-source heat pumps have improved low-temperature performance, and NREL reports that the technology can save energy for many cold-climate homes; this supports considering them as a viable option, not assuming uniform results for every building. [https://research-hub.nlr.gov/en/publications/field-validation-of-air-source-heat-pumps-for-cold-climates]
- A DOE-funded NREL field study monitored 12 centrally ducted, variable-speed air-source heat pumps in single-family homes for the 2021–22 winter. Its limited sample is useful real-world evidence, but should not be treated as a citywide performance guarantee. [https://heatpumpdata.energy.gov/data/studies/nrel-field-validation-air-source-heat-pumps-cold-climates-20

### Try it
Change `question` and uncomment the call.

In [10]:
question = "Write a brief on EV battery recycling in Europe."  # your own request
# result, log = chat(question, "try-1")

### Done when
- The quick question did not read any file; the brief request read `SKILL.md` first, then `brief_template.md`.
- `output/workspace-part1/brief.md` has the four sections of the template and a url after each finding.